# Modelo de elementos finitos en OpenSeesPy de un puente de tres vanos (tipo Z24)

Objetivo: generar series sintéticas de **frecuencia natural y temperatura** de un puente sano y dañado, para el proyecto de confiabilidad (alarma, medición con temperatura y monitoreo continuo).

Pasos del cuaderno:

1. Geometría y propiedades (elementos `elasticBeamColumn`, 2D, solo flexión vertical).
2. Construcción del modelo en OpenSeesPy.
3. Análisis de valores propios → frecuencias y formas modales.
4. Calibración contra las frecuencias medidas del Z24.
5. Daño como reducción de rigidez de elementos.
6. Efecto de la temperatura (relación bilineal).
7. Simulación de un año hora por hora, con error de identificación.

**Valores de referencia del Z24** (frecuencias experimentales, tabla de calibración de modelo en core.ac.uk/download/55631114.pdf): modo 1 = 3.89 Hz (flexión pura), modo 4 = 10.30 Hz (flexión no simétrica). Los modos 2, 3 y 5 son longitudinales o transversales y un modelo 2D vertical no los reproduce.

> Todos los valores marcados como `ASSUMED` son supuestos para arrancar y deben reemplazarse o calibrarse.

In [ ]:
# Install once if needed: pip install openseespy
import numpy as np
import openseespy.opensees as ops
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
rng = np.random.default_rng(42)

## 1. Geometría y propiedades

Vanos de 14, 30 y 14 m (58 m en total). Cada vano se divide en elementos de igual longitud.

- **Apoyos:** se restringe el desplazamiento vertical en los 4 apoyos; la rotación queda libre (viga continua). En el Z24 real los extremos tienen rótulas de concreto y las pilas están empotradas a la viga; aquí se simplifica.
- **Desplazamiento horizontal:** se restringe en todos los nodos para que el modelo solo tenga modos de flexión vertical (sin modos axiales).
- **Masa:** masa por metro del tablero (concreto + asfalto + barandas), `ASSUMED`.
- **Rigidez:** $EI$ se calibra para que el modo 1 dé 3.89 Hz.

In [ ]:
# --- Geometry ---
spans = [14.0, 30.0, 14.0]          # m
n_el_per_m = 1.0                    # elements per metre -> 14, 30, 14 elements

# --- Properties (ASSUMED, to be calibrated) ---
m_bar = 12_000.0                    # kg/m, deck mass per unit length (ASSUMED)
E_ref = 35e9                        # Pa, concrete modulus at reference temperature (ASSUMED)
I_guess = 3.0                       # m^4, initial guess of box-girder inertia (ASSUMED)
A_sec = 5.0                         # m^2, cross-section area (irrelevant here: horizontal DOF fixed)

# Mesh
x_nodes = [0.0]
support_x = [0.0]
for L in spans:
    n = int(round(L * n_el_per_m))
    x0 = x_nodes[-1]
    x_nodes += list(x0 + np.linspace(0, L, n + 1)[1:])
    support_x.append(x_nodes[-1])
x_nodes = np.array(x_nodes)
n_nodes = len(x_nodes)
n_el = n_nodes - 1
support_nodes = [int(np.argmin(abs(x_nodes - s))) for s in support_x]
pier_nodes = support_nodes[1:-1]

print(f"Nodes: {n_nodes}, elements: {n_el}")
print("Support nodes:", support_nodes, "at x =", x_nodes[support_nodes])

## 2. Modelo en OpenSeesPy

- `ops.model('basic', '-ndm', 2, '-ndf', 3)`: modelo plano con 3 grados de libertad por nodo: $u_x$, $u_y$ (vertical) y $\theta_z$ (giro).
- `ops.fix(node, ux, uy, rz)`: 1 = restringido, 0 = libre.
- `elasticBeamColumn` con `-mass` $\bar m$ y `-cMass`: viga elástica de Euler-Bernoulli con **masa consistente**.

Cada elemento recibe un módulo $E_e=E\,(1-d_e)$, donde $d_e$ es la **pérdida de rigidez** del elemento (0 = sano, 0.05 = 5 %). Como $I$ queda fijo, reducir $E$ equivale a reducir la rigidez a flexión $EI$.

El modelo se reconstruye desde cero (`ops.wipe()`) cada vez que cambia el daño.

In [ ]:
def build_model(E, I, damage=None):
    # damage: array of length n_el with stiffness loss d_e in [0, 1)
    if damage is None:
        damage = np.zeros(n_el)
    ops.wipe()
    ops.model("basic", "-ndm", 2, "-ndf", 3)
    for n, x in enumerate(x_nodes):
        ops.node(n + 1, float(x), 0.0)
        uy_fixed = 1 if n in support_nodes else 0
        ops.fix(n + 1, 1, uy_fixed, 0)        # ux fixed everywhere, uy fixed at supports
    ops.geomTransf("Linear", 1)
    for e in range(n_el):
        E_e = E * (1 - damage[e])
        ops.element("elasticBeamColumn", e + 1, e + 1, e + 2,
                    A_sec, E_e, I, 1, "-mass", m_bar, "-cMass")

## 3. Análisis de valores propios

Se resuelve el problema generalizado
$$
\mathbf{K}\,\boldsymbol{\phi}=\omega^2\,\mathbf{M}\,\boldsymbol{\phi},\qquad f=\frac{\omega}{2\pi}
$$

- $\mathbf{K}$, $\mathbf{M}$: matrices globales de rigidez y masa (OpenSees las arma internamente).

`ops.eigen(n)` devuelve los valores propios $\lambda=\omega^2$, y `ops.nodeEigenvector(node, mode, dof)` la forma modal en cada nodo.
- $\omega$: frecuencia circular (rad/s); $f$: frecuencia natural (Hz).
- $\boldsymbol{\phi}$: forma modal.

In [ ]:
def modal(E, I, damage=None, n_modes=4):
    build_model(E, I, damage)
    lam = np.array(ops.eigen(n_modes))
    freqs = np.sqrt(lam) / (2 * np.pi)
    # vertical component of each mode shape at every node
    phi = np.array([[ops.nodeEigenvector(n + 1, k + 1, 2) for k in range(n_modes)]
                    for n in range(n_nodes)])
    return freqs, phi

f_guess, _ = modal(E_ref, I_guess)
print("Frequencies with initial guess (Hz):", np.round(f_guess, 3))

## 4. Calibración

Como $f\propto\sqrt{EI/\bar m}$, la inercia que hace que el modo 1 dé el valor medido es
$$
I_{cal}=I_{guess}\left(\frac{f_{1,target}}{f_{1,guess}}\right)^2
$$

Después se compara el segundo modo vertical del modelo con el modo 4 del Z24 (10.30 Hz).

**Resultado esperado** (verificado con un modelo equivalente de matrices propias): $I_{cal}\approx 0.673$ m⁴ y frecuencias ≈ 3.89, 10.39, 13.92 y 16.51 Hz; el segundo modo queda 0.9 % por encima de los 10.30 Hz medidos.

In [ ]:
f1_target = 3.89     # Hz, Z24 mode 1 (vertical bending)
f2_target = 10.30    # Hz, Z24 mode 4 (non-symmetric vertical bending)

I_cal = I_guess * (f1_target / f_guess[0])**2
EI_cal = E_ref * I_cal
f_cal, phi_cal = modal(E_ref, I_cal)

print(f"Calibrated I = {I_cal:.3f} m^4  (EI = {EI_cal:.3e} N*m^2)")
print("Model frequencies (Hz):", np.round(f_cal, 3))
print(f"Mode 2 (vertical) vs Z24 mode 4: {f_cal[1]:.2f} vs {f2_target} Hz "
      f"({100*(f_cal[1]/f2_target - 1):+.1f} %)")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 6), sharex=True)
for i, ax in enumerate(axes):
    w = phi_cal[:, i]
    ax.plot(x_nodes, w / np.max(np.abs(w)), "b-", lw=2)
    ax.axhline(0, color="k", lw=0.5)
    ax.plot(x_nodes[support_nodes], np.zeros(4), "k^", ms=10)
    ax.set_ylabel(f"Mode {i+1}\n{f_cal[i]:.2f} Hz")
axes[-1].set_xlabel("x (m)")
fig.suptitle("Vertical bending mode shapes (healthy, calibrated)")
plt.tight_layout(); plt.show()

## 5. Daño

Función que asigna una pérdida de rigidez $d$ a los elementos de un tramo $[x_a, x_b]$. Ejemplo: daño cerca de la pila del lado de Koppigen (en el Z24 el primer escenario de daño fue el descenso de esa pila).

Se calcula la caída de frecuencia de los modos 1 y 2 para varios niveles de daño.

In [ ]:
def damage_vector(x_a, x_b, d):
    xm = 0.5 * (x_nodes[:-1] + x_nodes[1:])        # element midpoints
    dam = np.zeros(n_el)
    dam[(xm >= x_a) & (xm <= x_b)] = d
    return dam

dmg_zone = (40.0, 48.0)          # m, zone near the second pier (ASSUMED location)
d_levels = [0.0, 0.02, 0.05, 0.10, 0.20, 0.30]

rows = []
for d in d_levels:
    fd, _ = modal(E_ref, I_cal, damage_vector(*dmg_zone, d), n_modes=2)
    rows.append(fd)
rows = np.array(rows)
for d, fd in zip(d_levels, rows):
    print(f"d = {d:4.2f}: f1 = {fd[0]:.4f} Hz ({100*(fd[0]/rows[0,0]-1):+.3f} %), "
          f"f2 = {fd[1]:.4f} Hz ({100*(fd[1]/rows[0,1]-1):+.3f} %)")

## 6. Efecto de la temperatura

En el Z24 la frecuencia variaba casi linealmente con la temperatura por encima de 0 °C y con una pendiente mucho más empinada por debajo (asfalto congelado; Peeters y De Roeck, 2001). Se representa con un factor bilineal sobre el módulo:
$$
\frac{E(T)}{E_{ref}}=
\begin{cases}
1+a_w\,(T-T_{ref}), & T\ge 0\\[2pt]
1+a_w\,(0-T_{ref})+a_c\,T, & T<0
\end{cases}
$$

- $T$: temperatura (°C); $T_{ref}$: temperatura de referencia.
- $a_w$: pendiente en clima templado (por °C, negativa: más calor → menos rigidez).
- $a_c$: pendiente bajo cero (más negativa: más frío → mucho más rigidez).

Como la temperatura escala la rigidez de todo el puente por igual, la frecuencia escala con la raíz:
$$
f(T,d)=f(T_{ref},d)\,\sqrt{E(T)/E_{ref}}
$$
Así solo hace falta un análisis modal por nivel de daño, no uno por hora.

> `a_w` y `a_c` son `ASSUMED`; hay que calibrarlos con la figura frecuencia–temperatura de Peeters y De Roeck (2001).

In [ ]:
T_ref = 10.0        # degC (ASSUMED)
a_w = -0.002        # 1/degC above 0 degC (ASSUMED)
a_c = -0.015        # 1/degC below 0 degC (ASSUMED)

def E_factor(T):
    T = np.asarray(T, dtype=float)
    warm = 1 + a_w * (T - T_ref)
    cold = 1 + a_w * (0 - T_ref) + a_c * T
    return np.where(T >= 0, warm, cold)

T_plot = np.linspace(-20, 35, 200)
plt.figure(figsize=(7, 4))
plt.plot(T_plot, f_cal[0] * np.sqrt(E_factor(T_plot)), "b-", label="Healthy")
fd5 = modal(E_ref, I_cal, damage_vector(*dmg_zone, 0.05), n_modes=1)[0][0]
plt.plot(T_plot, fd5 * np.sqrt(E_factor(T_plot)), "r--", label="d = 5 %")
plt.axvline(0, color="k", lw=0.5)
plt.xlabel("Temperature (°C)"); plt.ylabel("f1 (Hz)")
plt.title("Bilinear frequency–temperature model (mode 1)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 7. Simulación de un año, hora por hora

Modelo de la frecuencia medida:
$$
f_{med}=f(T,d)+e_{id},\qquad e_{id}\sim N(0,\sigma_{id}^2)
$$

- $e_{id}$: error de identificación modal, normal con media cero (`sigma_id` es `ASSUMED`; buscar valor en la literatura de análisis modal operacional).

**Temperatura:** aquí se usa una serie **sintética** (ciclo anual + ciclo diario + ruido) solo para que el cuaderno corra. Se debe reemplazar por la serie horaria real de Environment Canada (aeropuerto de Edmonton) cargando el CSV en `T_hourly`.

**Daño:** el año 1 es sano (para construir la PDF del puente sano). En el año 2 el daño aparece en `t_onset` y crece linealmente hasta `d_final` (etapa 3, daño incipiente).

In [ ]:
hours = np.arange(365 * 24)

def synthetic_temperature(h, rng):
    # Placeholder: replace with Environment Canada hourly data
    annual = -12.0 * np.cos(2 * np.pi * h / (365 * 24))    # ASSUMED amplitude
    daily = -5.0 * np.cos(2 * np.pi * h / 24)              # ASSUMED amplitude
    return 3.0 + annual + daily + rng.normal(0, 3.0, h.size)

T_hourly = synthetic_temperature(hours, rng)
sigma_id = 0.01      # Hz (ASSUMED)

# Frequency at T_ref as a function of damage, by interpolation over a grid
d_grid = np.linspace(0, 0.30, 31)
f1_grid = np.array([modal(E_ref, I_cal, damage_vector(*dmg_zone, d), n_modes=1)[0][0]
                    for d in d_grid])

def f1_measured(T, d, rng):
    f_ref = np.interp(d, d_grid, f1_grid)
    return f_ref * np.sqrt(E_factor(T)) + rng.normal(0, sigma_id, np.size(T))

# Year 1: healthy
f_year1 = f1_measured(T_hourly, np.zeros(hours.size), rng)

# Year 2: progressive damage
t_onset = 180 * 24                  # hour at which damage starts (ASSUMED)
d_final = 0.10                      # damage at the end of the year (ASSUMED)
d_hist = np.clip((hours - t_onset) / (hours[-1] - t_onset), 0, 1) * d_final
T_year2 = synthetic_temperature(hours, rng)
f_year2 = f1_measured(T_year2, d_hist, rng)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
days = hours / 24
axes[0].plot(days, T_year2, lw=0.4); axes[0].set_ylabel("T (°C)")
axes[1].plot(days, 100 * d_hist, "r"); axes[1].set_ylabel("Damage d (%)")
axes[2].plot(days, f_year2, lw=0.4); axes[2].set_ylabel("f1 measured (Hz)")
axes[2].set_xlabel("Day of year 2")
fig.suptitle("Synthetic year with progressive damage")
plt.tight_layout(); plt.show()

plt.figure(figsize=(7, 4.5))
plt.scatter(T_hourly, f_year1, s=1, alpha=0.3, label="Year 1, healthy")
mask = d_hist > 0.05
plt.scatter(T_year2[mask], f_year2[mask], s=1, alpha=0.3, c="r", label="Year 2, d > 5 %")
plt.xlabel("Temperature (°C)"); plt.ylabel("f1 (Hz)")
plt.legend(markerscale=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# Save the synthetic data for the reliability analysis
import csv
with open("synthetic_bridge_data.csv", "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["year", "hour", "T_degC", "damage", "f1_Hz"])
    for h in hours:
        w.writerow([1, h, round(T_hourly[h], 3), 0.0, round(f_year1[h], 5)])
    for h in hours:
        w.writerow([2, h, round(T_year2[h], 3), round(d_hist[h], 5), round(f_year2[h], 5)])
print("Saved synthetic_bridge_data.csv")